# Database Production Engineering
### N+1 Queries | Connection Pooling | Transactional Outbox | Indexing | Migrations

> **System:** ShopFlow -- PostgreSQL backend, 500k daily users.
> Format: **Mental Model -> Real Scenario -> BEFORE -> AFTER -> Frameworks -> Nuances**

*Shift+Enter to run each cell*

## Setup

In [ ]:
from __future__ import annotations
import time, threading, uuid
from contextlib import contextmanager
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Any
print('Setup OK')

---
## 1 · The N+1 Query Problem

### Mental Model -- 'One Trip vs N Trips to the Library'

```
WHAT   Fetching N parent records then making 1 query PER parent to get
       related data = N+1 total queries (1 for the list + N for details).
WHY    Each query has network round-trip overhead. N=100 orders =
       101 queries. N=10,000 = 10,001 queries. Linear scaling = O(N) DB hits.
HOW    Replace with a JOIN or a batch IN(...) query: 2 queries total.
WHEN   Any time you loop over a queryset and access a relationship.
       The ORM makes this invisible until you turn on query logging.
```

### Nuance 1: N+1 is invisible without query logging
In development, N=10 so 11 queries feels fast. In production, N=500 means
501 queries. Always log slow queries AND query counts in production.
SQLAlchemy: `echo=True`. Django: `django-debug-toolbar` or `connection.queries`.

### Nuance 2: Eager loading can cause N+1 in reverse
SELECT * FROM orders JOIN products -- if each order has 50 products, you're
loading 50x the data needed. Use `LIMIT` + `JOIN` carefully, or load
products separately with `IN(order_ids)` (2 queries, not N+1, not 50x).

### Nuance 3: N+1 in async code is especially damaging
In async FastAPI, each query is an await. N awaits in a loop = N round trips
with no concurrency benefit. Use `asyncio.gather()` for truly concurrent
parallel queries, or better, a single batch query.

### Real-World Scenario -- ShopFlow Order List API

**Incident:** The admin dashboard `/orders?status=shipped` loaded 200 orders.
For each order it fetched the customer name: 200 extra queries.
The page took 4.2 seconds. DB CPU jumped to 85% during each admin login.

**Fix:** Replace the loop with a single JOIN query.
Page load: 4.2s -> 0.08s. DB CPU: 85% -> 3%.

In [ ]:
# Simulated DB (counts every query made)

@dataclass
class FakeDB:
    query_count: int = 0
    query_log:   list[str] = field(default_factory=list)

    def query(self, sql: str, **params) -> list[dict]:
        self.query_count += 1
        self.query_log.append(sql)
        return []  # results not needed for this demo


db = FakeDB()

# Simulated data: 100 orders, each with a customer_id
orders = [{'id': i, 'customer_id': (i % 20) + 1, 'total': 9.99} for i in range(1, 101)]

# BEFORE -- N+1: 1 query for orders + 1 query per order for customer
def get_orders_with_customers_BAD(db: FakeDB) -> list[dict]:
    db.query('SELECT * FROM orders WHERE status = shipped')  # query 1
    result = []
    for order in orders:
        # query 2...N+1: one per order
        db.query('SELECT name FROM customers WHERE id = :id', id=order['customer_id'])
        result.append({**order, 'customer_name': 'Alice'})
    return result

db_bad = FakeDB()
get_orders_with_customers_BAD(db_bad)
print(f'BEFORE (N+1): {db_bad.query_count} queries for {len(orders)} orders')
print(f'  First 3 queries: {db_bad.query_log[:3]}')

In [ ]:
# AFTER -- Fix 1: JOIN (single query)
def get_orders_with_customers_JOIN(db: FakeDB) -> list[dict]:
    db.query('SELECT o.*, c.name FROM orders o JOIN customers c ON c.id = o.customer_id')
    return [{'id': o['id'], 'customer_name': 'Alice'} for o in orders]

db_join = FakeDB()
get_orders_with_customers_JOIN(db_join)
print(f'AFTER (JOIN): {db_join.query_count} query total')

# AFTER -- Fix 2: Batch IN query (2 queries, useful when join is complex)
def get_orders_with_customers_BATCH(db: FakeDB) -> list[dict]:
    db.query('SELECT * FROM orders WHERE status = shipped')   # query 1
    customer_ids = list({o['customer_id'] for o in orders})
    db.query(f'SELECT id, name FROM customers WHERE id IN ({customer_ids})')  # query 2
    # build a map: customer_id -> name
    return [{'id': o['id'], 'customer_name': 'Alice'} for o in orders]

db_batch = FakeDB()
get_orders_with_customers_BATCH(db_batch)
print(f'AFTER (BATCH): {db_batch.query_count} queries total -- constant, not O(N)')
print(f'Reduction: {db_bad.query_count} -> {db_batch.query_count} queries')

### Where This Is Seen in Real Frameworks

| Framework | N+1 detection & fix |
|-----------|---------------------|
| **SQLAlchemy** | `joinedload(Order.customer)` or `selectinload()` -- specify eagerly; `lazy='raise'` to catch N+1 at test time |
| **Django ORM** | `select_related('customer')` (JOIN) or `prefetch_related('items')` (IN batch) |
| **Strawberry/Graphene** | DataLoader pattern batches N field resolves into 1 DB query |
| **django-debug-toolbar** | Shows query count per request; N+1 is immediately visible |
| **SQLAlchemy `echo=True`** | Logs every SQL statement; enables query count auditing |

---
## 2 · Connection Pooling -- The DB's Finite Resource

### Mental Model -- 'The Shared Printer'

```
WHAT   Maintain a fixed set of open DB connections; reuse them across requests.
WHY    Opening a DB connection = TCP handshake + auth + session setup (~5-15ms).
       PostgreSQL default max_connections=100. 1000 requests/s without pooling
       = 1000 connections = 'FATAL: too many connections'.
HOW    A pool checks out an existing connection, runs the query, checks it back in.
       A context manager ALWAYS returns the connection, even on exception.
WHEN   Every production application with a relational DB.
       Rule of thumb: pool_size = (num_workers * 2) + 1 per worker process.
```

### Nuance 1: Pool size is NOT 'more is better'
PostgreSQL has a hard limit. With 8 Gunicorn workers, each needing a pool of 5,
you need 40 connections. Set `max_connections=100` in Postgres means
max 2 apps x 8 workers x 5 pool = 80. PgBouncer is a pool proxy that
multiplexes thousands of app connections to a few real DB connections.

### Nuance 2: Connection leaks are catastrophic and silent
A leaked connection (exception path that skips close()) holds a slot forever.
The pool gradually drains; the app hangs when the pool is empty. Always use
a context manager: `with pool.acquire() as conn:` guarantees release.

### Nuance 3: Pre-ping prevents stale connections
DB servers close idle connections after a timeout. A connection that was
idle in the pool for hours is stale. `pool_pre_ping=True` (SQLAlchemy) sends
a cheap `SELECT 1` before returning the connection, recycling dead ones.

In [ ]:
from contextlib import contextmanager

class PoolExhausted(Exception): pass


@dataclass
class Connection:
    id: int
    in_use: bool = False


class ConnectionPool:
    def __init__(self, max_size: int, wait_timeout: float = 0.1):
        self.max_size     = max_size
        self._conns       = [Connection(i) for i in range(max_size)]
        self._free        = list(self._conns)
        self._lock        = threading.Lock()
        self.wait_timeout = wait_timeout
        self.opens        = max_size   # connections physically opened (once)
        self.rejections   = 0

    @property
    def in_use(self): return sum(c.in_use for c in self._conns)

    @contextmanager
    def acquire(self):
        # Try to get a free connection with a brief wait
        deadline = time.monotonic() + self.wait_timeout
        conn = None
        while conn is None:
            with self._lock:
                if self._free:
                    conn = self._free.pop()
                    conn.in_use = True
            if conn is None:
                if time.monotonic() > deadline:
                    self.rejections += 1
                    raise PoolExhausted(f'No free connection (max={self.max_size})')
                time.sleep(0.001)
        try:
            yield conn
        finally:
            with self._lock:
                conn.in_use = False
                self._free.append(conn)  # ALWAYS returned


pool = ConnectionPool(max_size=3)

# 50 sequential requests reuse the same 3 connections
for _ in range(50):
    with pool.acquire() as conn:
        assert conn.in_use

print(f'50 requests, only {pool.opens} connections physically opened')

# Simulate leak: don't use context manager -> connection never returned
leaked: list = []
try:
    for _ in range(3): leaked.append(pool._free.pop())  # drain pool
    with pool.acquire() as c: pass  # should raise PoolExhausted
except PoolExhausted as e:
    print(f'Leak simulation: {e}')
    print(f'Rejections: {pool.rejections}')
finally:
    for c in leaked: pool._free.append(c)  # restore for cleanup

### Where This Is Seen in Real Frameworks

| Framework | Pooling mechanism |
|-----------|------------------|
| **SQLAlchemy** | `create_engine(url, pool_size=5, max_overflow=10, pool_pre_ping=True)` |
| **asyncpg** | `asyncpg.create_pool(min_size=5, max_size=20)` |
| **Django** | `DATABASES['default']['CONN_MAX_AGE'] = 60` (persistent connections per worker) |
| **PgBouncer** | Transaction-mode pooling: 1000 app connections -> 10 real DB connections |
| **Redis** | `ConnectionPool(max_connections=50)` -- same pattern, different storage |

---
## 3 · Transactional Outbox -- The Dual-Write Problem

### Mental Model -- 'Leave a Note Before Calling'

```
WHAT   Write the business record AND an 'outbox' event to the SAME DB
       transaction. A separate relay publishes outbox events to the message
       broker, marking each as sent after confirmed publish.
WHY    Dual-write (write to DB then publish to Kafka) has no atomicity.
       A crash between the two leaves DB and Kafka out of sync:
       order exists but no event fired, or event fired but order rolled back.
HOW    Outbox table in same DB. Atomic: commit order + outbox row together.
       Relay: poll outbox WHERE sent=false, publish, UPDATE sent=true.
WHEN   Any time a service writes to a DB AND publishes an event.
```

### Nuance 1: The relay must use at-least-once semantics
If the relay crashes between publish and UPDATE sent=true, it republishes.
Consumers MUST be idempotent (dedupe by event_id) -- you will get duplicates.

### Nuance 2: Polling vs CDC
Polling the outbox (SELECT WHERE sent=false) is simple but adds DB load.
Change Data Capture (Debezium reads Postgres WAL) is push-based: zero polling,
sub-second latency, but more operational complexity.

### Nuance 3: Outbox rows must be pruned
Sent outbox rows accumulate indefinitely. Run a nightly job:
`DELETE FROM outbox WHERE sent=true AND created_at < NOW() - INTERVAL '7 days'`.
Without pruning, the outbox table grows forever and slows the relay.

In [ ]:
# Transactional Outbox Pattern

@dataclass
class Order:
    id:    str
    total: float
    email: str


@dataclass
class OutboxEvent:
    id:         str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    event_type: str = ''
    payload:    dict = field(default_factory=dict)
    sent:       bool = False


class FakeDatabase:
    def __init__(self):
        self.orders: dict[str, Order]       = {}
        self.outbox: list[OutboxEvent]       = []
        self._in_tx: bool = False
        self._tx_buffer: list = []

    @contextmanager
    def transaction(self):
        self._in_tx = True
        self._tx_buffer.clear()
        try:
            yield self
            # Commit -- apply all buffered writes atomically
            for op, args in self._tx_buffer:
                op(*args)
            print('  [DB] COMMIT: order + outbox row written atomically')
        except Exception:
            self._tx_buffer.clear()
            print('  [DB] ROLLBACK: nothing persisted')
            raise
        finally:
            self._in_tx = False

    def save_order(self, order: Order):
        self._tx_buffer.append((lambda o: self.orders.update({o.id: o}), [order]))

    def save_outbox(self, event: OutboxEvent):
        self._tx_buffer.append((lambda e: self.outbox.append(e), [event]))


class FakeBroker:
    def __init__(self): self.published: list[dict] = []
    def publish(self, event: OutboxEvent) -> None:
        self.published.append({'id': event.id, 'type': event.event_type})
        print(f'  [Broker] Published: {event.event_type} ({event.id})')


db     = FakeDatabase()
broker = FakeBroker()

# BEFORE -- Dual-write: order saved to DB, then published to broker
# A crash between the two = LOST EVENT or GHOST EVENT
print('BEFORE (dual-write -- dangerous):')
order = Order(id='ord-001', total=49.99, email='alice@shopflow.com')
db.orders['ord-001'] = order     # write to DB
# << crash here = order exists but no event published >>
broker.publish(OutboxEvent(event_type='order.placed', payload={'id': 'ord-001'}))
print(f'  DB orders: {list(db.orders.keys())}')
print(f'  Broker events: {len(broker.published)} -- out of sync if crashed between')

In [ ]:
# AFTER -- Transactional Outbox

def place_order_safe(db: FakeDatabase, broker: FakeBroker, order: Order) -> None:
    event = OutboxEvent(event_type='order.placed', payload={'id': order.id})
    with db.transaction():
        db.save_order(order)   # both writes in ONE transaction
        db.save_outbox(event)  # atomic -- either both commit or both rollback

def relay(db: FakeDatabase, broker: FakeBroker) -> None:
    # Polls outbox for unsent events; publishes and marks sent
    for event in db.outbox:
        if not event.sent:
            broker.publish(event)
            event.sent = True   # mark AFTER confirmed publish
            # Real: UPDATE outbox SET sent=true WHERE id=:id


db2     = FakeDatabase()
broker2 = FakeBroker()

print('AFTER (transactional outbox):')
order2 = Order('ord-002', 99.99, 'bob@shopflow.com')
place_order_safe(db2, broker2, order2)
print(f'  DB orders: {list(db2.orders.keys())}')
print(f'  Outbox (unsent): {[e.id for e in db2.outbox if not e.sent]}')

print('\nRelay running...')
relay(db2, broker2)
print(f'  Outbox (all sent): {all(e.sent for e in db2.outbox)}')
print(f'  Broker events: {broker2.published}')
print('Order and event are ALWAYS in sync')

### Where This Is Seen in Real Frameworks

| Tool | Outbox implementation |
|------|----------------------|
| **Debezium** | CDC reads Postgres WAL -- publishes without polling |
| **Outboxer (Node.js)** | Polls outbox table, publishes to Kafka/RabbitMQ |
| **SQLAlchemy** | `session.add(order); session.add(outbox_event); session.commit()` |
| **Django** | `with transaction.atomic(): order.save(); OutboxEvent.objects.create(...)` |
| **Postgres LISTEN/NOTIFY** | Trigger notifies relay on outbox insert -- lower latency than polling |

---
## 4 · Indexing -- The 10,000x Query Difference

### Mental Model -- 'The Book Index vs Reading Every Page'

```
WHAT   A B-tree index on a column lets the DB find rows in O(log N)
       instead of scanning all N rows (O(N) -- a 'full table scan').
WHY    On 10M rows: full scan = 10M row reads; index seek = ~24 reads.
       10M / 24 = ~416,000x cheaper on that column.
HOW    CREATE INDEX ON orders(status, created_at DESC); -- composite index.
       WHERE status='shipped' ORDER BY created_at DESC -- uses index.
WHEN   Any column in WHERE, JOIN ON, ORDER BY, GROUP BY.
       Use EXPLAIN ANALYZE to confirm the index is actually used.
```

### Nuance 1: Index column order matters
Composite index (a, b): WHERE a=? uses it; WHERE b=? does NOT (leading column rule).
WHERE a=? AND b=? uses it. WHERE a=? ORDER BY b uses it. Design for your queries.

### Nuance 2: Indexes slow writes
Every INSERT/UPDATE/DELETE must also update every index on the table.
A table with 10 indexes has 10x the write overhead. Drop unused indexes.
On bulk loads: drop indexes, load, rebuild (much faster).

### Nuance 3: Partial indexes reduce index size dramatically
`CREATE INDEX ON orders(created_at) WHERE status='pending'` -- only indexes
pending orders. If 99% of orders are 'delivered', the index is 100x smaller
and queries on pending orders are faster.

In [ ]:
import bisect

# Simulate a table and index to show scan cost difference

@dataclass(frozen=True)
class DBRow:
    id:         int
    status:     str
    created_at: int  # epoch seconds


class SimulatedTable:
    def __init__(self, n: int):
        statuses = ['pending', 'shipped', 'delivered', 'cancelled']
        self.rows = [
            DBRow(i, statuses[i % 4], 1_700_000_000 + i)
            for i in range(n)
        ]
        # Build B-tree index on status (simulated as a sorted dict of id lists)
        self._index: dict[str, list[int]] = defaultdict(list)
        for row in self.rows:
            self._index[row.status].append(row.id)
        self.last_scan = 0

    def full_scan(self, status: str) -> list[DBRow]:
        result = []
        self.last_scan = 0
        for row in self.rows:          # reads EVERY row
            self.last_scan += 1
            if row.status == status:
                result.append(row)
        return result

    def index_seek(self, status: str) -> list[DBRow]:
        ids = set(self._index.get(status, []))
        self.last_scan = len(ids)      # only reads matching rows
        return [r for r in self.rows if r.id in ids]


table = SimulatedTable(100_000)

# Full table scan (no index)
full_results = table.full_scan('pending')
full_cost = table.last_scan

# Index seek
idx_results = table.index_seek('pending')
idx_cost = table.last_scan

print(f'Table: 100,000 rows, querying WHERE status=pending')
print(f'  Full scan rows read: {full_cost:,}')
print(f'  Index seek rows read: {idx_cost:,}')
print(f'  Speedup: ~{full_cost // max(idx_cost,1)}x')
print(f'  Both return {len(full_results)} rows')

### Where This Is Seen in Real Frameworks

| Tool | Indexing |
|------|----------|
| **PostgreSQL EXPLAIN** | `EXPLAIN (ANALYZE, BUFFERS) SELECT ...` -- see Seq Scan vs Index Scan |
| **SQLAlchemy** | `Index('ix_orders_status', Order.status)` in model or migration |
| **Django** | `class Meta: indexes = [models.Index(fields=['status', 'created_at'])]` |
| **pg_stat_user_tables** | `seq_scan` column -- high number means a missing index |
| **Alembic / Django migrations** | `op.create_index('ix_...')` -- index creation is a DDL migration |

---
## 5 · Zero-Downtime Migrations -- The Lock Problem

### Mental Model -- 'Rebuilding the Runway While Planes Land'

```
WHAT   Schema changes on large tables acquire table locks that block
       all reads and writes -- effectively taking the site down.
WHY    ALTER TABLE orders ADD COLUMN ship_date TIMESTAMP locks the
       table for the duration (minutes on a 100M-row table in Postgres <11).
HOW    Expand-contract pattern:
       Step 1: Add nullable column (no default = lock-free in PG12+)
       Step 2: Backfill in small batches (no table lock)
       Step 3: Deploy code that reads new column
       Step 4: Add NOT NULL constraint after all rows backfilled
WHEN   Adding columns, adding indexes, adding NOT NULL constraints
       to tables with > ~100k rows in a live system.
```

### Nuance 1: CREATE INDEX CONCURRENTLY
Plain `CREATE INDEX` locks the table. `CREATE INDEX CONCURRENTLY` does not.
It takes longer and can fail (then leaves an invalid index to clean up),
but it's the only option on a live table > ~50k rows.

### Nuance 2: NOT NULL + DEFAULT in Postgres < 11 is deadly
`ALTER TABLE ADD COLUMN x INT NOT NULL DEFAULT 0` in PG < 11 rewrites
the entire table while holding an exclusive lock. In PG12+ with a constant
default it's instant (stored in catalog). Know your Postgres version.

### Nuance 3: Test migrations on a copy first
Restore production backup to a staging DB. Run the migration. Measure the
lock duration. If > 1 second, redesign. Never guess on prod.

In [ ]:
# Simulate expand-contract migration pattern

@dataclass
class MigrationStep:
    name:        str
    locks_table: bool
    duration_s:  float  # simulated

    def run(self) -> str:
        status = 'BLOCKING' if self.locks_table else 'safe'
        return f'[{status}] {self.name} ({self.duration_s:.1f}s)'


# BEFORE -- single ALTER TABLE migration (dangerous on large table)
dangerous_migration = [
    MigrationStep('ALTER TABLE orders ADD COLUMN ship_date TIMESTAMPTZ NOT NULL DEFAULT NOW()',
                  locks_table=True, duration_s=120),  # 2-min lock on 100M rows
]

print('BEFORE (dangerous):')
for step in dangerous_migration:
    print(' ', step.run())

# AFTER -- expand-contract pattern (zero downtime)
safe_migration = [
    MigrationStep('Step 1: ADD COLUMN ship_date TIMESTAMPTZ NULL',
                  locks_table=False, duration_s=0.01),  # instant in PG12+
    MigrationStep('Step 2: Backfill in batches (UPDATE ... WHERE id BETWEEN x AND y)',
                  locks_table=False, duration_s=60),    # runs offline, no lock
    MigrationStep('Step 3: Deploy code reading ship_date (handles NULL)',
                  locks_table=False, duration_s=0.0),
    MigrationStep('Step 4: ALTER COLUMN ship_date SET NOT NULL (after all rows filled)',
                  locks_table=False, duration_s=5),     # PG12+ is instant with check constraint
]

print('AFTER (expand-contract, zero downtime):')
for step in safe_migration:
    print(' ', step.run())

lock_minutes_before = sum(s.duration_s for s in dangerous_migration if s.locks_table) / 60
lock_minutes_after  = sum(s.duration_s for s in safe_migration if s.locks_table) / 60
print(f'\nTable lock time: {lock_minutes_before:.0f}min -> {lock_minutes_after:.0f}min')

### Where This Is Seen in Real Frameworks

| Tool | Safe migration |
|------|---------------|
| **Alembic** | `op.add_column(nullable=True)` then separate `op.alter_column(nullable=False)` |
| **Django** | `SeparateDatabaseAndState` for multi-step migrations |
| **strong_migrations (Rails)** | Lints for dangerous migrations automatically |
| **gh-ost (GitHub)** | Online schema change tool; replays binlog to copy table live |
| **pgroll** | Version-controlled, multi-step schema changes for Postgres |